`hive-video-examples.ipynb`

# Hive Video Examples

Peter Dresslar, 2026/09/07

Demonstrations of the tools available from the repository, https://github.com/Collective-Logic-Lab/honeybee-hive-video

## Setting Things Up

In [ ]:
# This cell will verify your environment
import sys
from pathlib import Path

print (sys.executable)

# The following code is used to make it possible to run this notebook from the project root or a notebook subdirectory.
working_dir = Path.cwd().resolve()
for project_root in (working_dir, *working_dir.parents):
    if (project_root / "notebooks" / "hive-video-examples.ipynb").is_file():
        break
else:
    raise FileNotFoundError(
        f"Cannot find honey-bee-behavior above {working_dir}. Open this notebook from the repository."
    )

data_dir = project_root / "data"
print(f"🐝 Data directory: {data_dir}")

We have a number of ways we can use the hive-video tools, but the easiest is to install the module from PyPi.

Below, we call the install with the optional add-on `resequence`, which is a larger, more complicated project than the other tools.
Your usage may need only `hive-video`, in which case you can leave out the bracketed extra.

The automatic resequencing command is awaiting release. For now, keep the working `honeybee-hive-video` checkout beside this repository and install it below. Restart the kernel first if you have already imported `hive_video`.


In [ ]:
hive_video_source = project_root.parent / "honeybee-hive-video"
%pip install -e "{hive_video_source}[resequence]"  # once, until the next package release
import hive_video

print(hive_video.__version__)


Your last three outputs should have indicated your kernel, your data directory, and the version of this notebook. If so, you are ready to continue.

### Setting up the CLI

In many cases you may want to just use the tools for one-off commands. You can do this using the `hive-video` command-line interface (CLI), which includes nearly identical functionality to the Python API explored below. We give examples in the sections that follow. 

To set up the CLI:

Open a terminal; a terminal opened from JupyterLab will work as well. If you use [`uv`](https://docs.astral.sh/uv/getting-started/installation/), you can install the tools with:

```bash
uv tool install "hive-video[resequence]"
hive-video --version
hive-video --help
```

This gives you a `hive-video` command that you can use from any directory. If `uv` reports that its tool directory is missing from your `PATH`, run `uv tool update-shell` and open a new terminal.

For a one-off command, `uvx` can fetch the package and run it in its own environment:

```bash
uvx --from "hive-video[resequence]" hive-video --help
```

You can replace `--help` with the command you want to run, such as `fragment --help`. The persistent installation command is `uv tool install`; `uvx` is the shortcut for running a tool on demand. The [uv tools guide](https://docs.astral.sh/uv/guides/tools/) covers both options.

If you prefer to use `pip`, that works too. From the project root, activate the virtual environment you are using for this notebook and install the package there:

```bash
source .venv/bin/activate
python -m pip install "hive-video[resequence]"
hive-video --version
hive-video --help
```

If you already ran the `%pip install` cell above in that environment, the CLI is already installed there. You only need to activate the environment in your terminal to use it.

As with the Python installation, you can leave out `[resequence]` if you only need downloading and fragments. The tools require Python 3.12 or newer.

> As a suggestion for managing Python versions on a workstation, check out the [pyenv](https://github.com/pyenv/pyenv) project. 🍯🍯


---
## Clipping videos with Fragment

To begin we have our fragment function. Since the videos are so large and so processor intensive to download or even view, it can be useful to manually locate and cut out "snippets" (fragments) for experimentation. The `fragment` tool is a versatile way to do this.

To see `fragment` in action, we have a small, ten-second video example located at `data/examples/resequenced_example.mp4`. Note that this particular video has been resequenced and has visual timecodes, useful for our examples below.

Let's say that to start, we would like to cut the start of the video into five one-second chunks. Here's how we can do that using the `hive-video` API:

In [ ]:
from hive_video.fragment import create_fragment

# Use the data directory located in the setup cell.
reseq_example = data_dir / "examples/resequenced_example.mp4"
output_dir = data_dir / "temp/fragments/seconds"
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists

dur = 1  # duration of 1 second each.
chunks = 5

for i in range(chunks):
    out_path = output_dir / f"clip_{i:02d}.mp4"     # name the file using the range iterator
    clip = create_fragment(
        str(reseq_example),
        str(out_path),
        start=i * dur,
        duration=dur,
        unit="seconds",
    )
    print(clip)

Your output files will appear, along with metadata, in the specified directory: in this case `/data/temp/fragments/`.

> **Note**: if you re-run this notebook without changing the file names, these functions will fail since they do not overwrite files. Feel free to dump the entire `fragments/` folder if you want to start again.

We might also decide that we want to work with precise frame numbers to take a video fragment. Notice that in the sample video we do have frame numbers at the top that we can work with. Instead of taking 1 second slices, let's take 25 frame slices, which should be equivalent to the prior example---all the videos are encoded at 25fps.

In [ ]:
# reseq_example is already set.
output_dir = data_dir / "temp/fragments/frame_count"
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists

dur = 25  # duration of 1 second each.
chunks = 5

for i in range(chunks):
    out_path = output_dir / f"clip_{i:02d}.mp4"
    clip = create_fragment(
        str(reseq_example),
        str(out_path),
        start=i * dur,
        duration=dur,
        unit="frames",
    )
    print(clip)

Again, your output files will appear at `/data/temp/fragments/`, this time in the `frame_count` subdirectory. Notice that the frame numbers listed in the video should align with your directive to take 25 at a time.

Finally, we can optionally choose to take individual frames as  .png files, rather than outputing a video file. Maybe we just want ten frames from our example video:


In [ ]:
# reseq_example is already set.
output_dir = data_dir / "temp/fragments/stills"
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists

# dur = 0  # no duration
chunks = 10

for i in range(chunks):
    out_path = output_dir / f"clip_{i:02d}.png"      # note file extension!
    clip = create_fragment(
        str(reseq_example),
        str(out_path),
        start=i,
        unit="seconds",
    )
    print(clip)

### Running `fragment` from the CLI

In many cases you might like the flexibility of snipping one file at a time. For this task, the CLI tool might be more useful. `hive-video` has a command-line interface that you can install and run whereever it is needed.

For instance, you might like to take a large raw file and snip out ten minutes starting at the 60-minute mark. Here, it is important to remember that you want to work with seconds:

```bash
hive-video fragment \
  --video /path/to/full_video.mp4 \
  --start-seconds 3600 \
  --duration-seconds 600 \
  --out data/temp/fragments/cli/ten_minutes.mp4
```

Replace `/path/to/full_video.mp4` with the path to your full recording. This example needs at least 70 minutes of video: we start at 60 minutes and keep the next ten. The output paths in these terminal examples are relative to the project root. If your terminal is in `notebooks/`, run `cd ..` first.

To try the CLI with our included example, we can take its first second:

```bash
hive-video fragment \
  --video data/examples/resequenced_example.mp4 \
  --start-seconds 0 \
  --duration-seconds 1 \
  --out data/temp/fragments/cli/one_second.mp4
```

Or we can select by frame number. This takes 25 frames, starting with frame 25, so the last included frame is 49:

```bash
hive-video fragment \
  --video data/examples/resequenced_example.mp4 \
  --start-frame 25 \
  --duration-frames 25 \
  --out data/temp/fragments/cli/frames_25_49.mp4
```

Finally, to save one frame as a PNG, we leave out the duration and give the output a `.png` extension:

```bash
hive-video fragment \
  --video data/examples/resequenced_example.mp4 \
  --start-frame 50 \
  --out data/temp/fragments/cli/frame_50.png
```

As with the API examples, frame numbering starts at zero. Each output comes with a `.json` metadata file, and an existing output or metadata file will cause the command to stop. These examples write into a separate `cli/` folder so they can be tried alongside the Python examples above.

There are a few other options you can explore with `hive-video fragment --help`.


## Resequencing the raw videos: they need it!

The raw Edmonds videos are stored in files where the frames are consistently out of order. This leads to "jumps" in the video every few minutes---on average the cuts are perhaps 10 minutes apart. Understanding the evolution of the day from start to stop thus can be greatly aided by a reprocessed file. We call this "resequencing

### How resequencing works

Our resequencing process really boils down to a cut-and-search method. First, we process the video into low-density squares of data, and then we assign each square a value representing the pixel intensities therein. It stands to reason that the actual next frame of the video should have similar "square" values. We detect the "jumps" with large numerical changes between two frames (in the source video).

Of course, there can be sudden behavior in the hive. In order to smooth this out, we average the values from the 10 frames on each side of "the cut" when we are searching for the frame on the "other side". We reorganize the video and create a QC version with only the 10 tested frames on each "side", joined together by a single frame of all green. We sometimes call this the "green-flash-video;" the highlighted discontinuity makes human QA much easier.

One call finds the cuts, orders the pieces, and checks the joins. If QC passes, it writes a captioned video. Otherwise, it gives us a short video to review.

We will use `data/examples/raw-example_for-reseq.mp4` (a special 10 second video for this purpose) and save our work under `data/temp/resequence/example_01/`. Choose a fresh folder name for each run. 


In [ ]:
from hive_video.resequence.workflow import run_resequence
from IPython.display import Video                             # Allows showing the video in the notebook

resequence_dir = data_dir / "temp/resequence/example_01"
result = run_resequence(data_dir / "examples/raw-example_for-reseq.mp4", resequence_dir, profile="edmond-2019-v1")  # The `edmond-2019-v1` profile supplies the QC settings for these 25 fps videos.
Video(result["video"] or result["review_video"], embed=True)


### If QC asks for a review

The resequence process The video above shows the flagged joins, marked in green. Check them out! (Note that this manual step will take some real human time on one of the actual video files.) If they look correct, record your review and finish:

```python
from hive_video.resequence.workflow import approve_resequence
result = approve_resequence(resequence_dir, reviewer="Your name", note="What you checked")
```

If a join still looks wrong, leave it unapproved and follow the [detailed review instructions](https://github.com/Collective-Logic-Lab/honeybee-hive-video/blob/main/docs/agent-generated/resequencing.md). Our five-second sample currently takes this review path.

Automatic QC checks the joins; it cannot guarantee chronology or find every missed cut. The output includes a frame map back to the source; the current renderer omits the last source frame (this may be fixed in the future.) ꙮꙮꙮ


### Running `resequence` from the CLI

From the repository root, with the notebook's `.venv` active:

```bash
hive-video resequence run --video data/examples/raw_example.mp4 --out-dir data/temp/resequence/cli_01 --profile edmond-2019-v1
```

If it asks for review, watch the indicated video. To approve those joins and finish:

```bash
hive-video resequence finish --out-dir data/temp/resequence/cli_01 --reviewer "Your name" --note "What you checked"
```


## Downloading Hive Videos from Edmond

Our last tool to try is the download tool. Of course, you can simply click on the files on the website and download them to your workspace in many cases. However, there may be cases where this tool might be more useful.

**🍯🐝 Videos: BIG 🍯🐝**

A couple of things to consider for the videos:

1. They are very large. Some of the videos are 125GB+ (!!) ... Most are around 25-35GB. That is for one mp4 file.
2. They are organized with a particular order:
- Day (sequential day of trial)
- Side
- Panel

So for instance the filename:
	
`start01__20190606_190340_side0_bottom.mp4`

...means day 1 (couning from day zero); side 0, and bottom panel. What `day` means is obvious, but you may be interested to know more about `side` and `panel`: an illustrative guide appears here: https://edmond.mpg.de/file.xhtml?fileId=238043&version=1.0. As a team we have generally been pursuing understanding of the *top* panel videos, as they have more space for flight. We have worked with both sides 0 and 1 in the past.

So, let's say we would like to acquire the video from day 56, side 0, and top panel: `start56__20190809_174417_side0_top.mp4`. Our download tool allows us to shorthand the video in case we do not have the exact date and time handy. **This video is about 25GB in size.**

In [ ]:
from hive_video.download import download_video

output_dir = data_dir / "temp/raw"
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists


# This downloads the full video and can take many minutes or hours.
print("Finding the vid...", flush=True) 
# source = download_video(day=56, side=0, panel="top", target=output_dir, on_message=print, progress_seconds=60)    # The last two options control how the API reports download progress:
                                                                                                                    # remember that this could generate a lot of text and different notebook programs will respond in different ways.


### Running `download` from the CLI 

This large download is likely to take a very long time. So long, in fact, that we might not want to run it at all from the `hive-video` api within a python script. Our alternative is to use the `hive-video` CLI, which--given the nature of the website and the filenames involved, can be very convenient. Running from the CLI would look like:

```bash
hive-video download --start 56 --side 0 --panel top --target data/raw/day56/ --timeout 120
```
Note that in the command line call we have added a timeout value. There are several options like this one: view them with

```bash
hive-video download --help
```

If we can execute downloads using the CLI, why have the option to use Python and the API? The answer is that it is particularly useful to use the python API for pipelining scripts on a job-controlled, high-performance computing environment like ASU's Sol.